# Exercises

In this section we have two exercises:
1. Implement the polynomial kernel.
2. Implement the multiclass C-SVM.

## Polynomial kernel

You need to extend the ``build_kernel`` function and implement the polynomial kernel if the ``kernel_type`` is set to 'poly'. The equation that needs to be implemented:
\begin{equation}
K=(X^{T}*Y)^{d}.
\end{equation}

In [2]:
def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    elif kernel_type == 'poly':
        degree = 2
        kernel = np.power(kernel, degree)
    return kernel

## Implement a multiclass C-SVM

Use the classification method that we used in notebook 7.3 and IRIS dataset to build a multiclass C-SVM classifier. Most implementation is about a function that will return the proper data set that need to be used for the prediction. You need to implement:
- ``choose_set_for_label``
- ``get_labels_count``

In [8]:
def choose_set_for_label(data_set, label):
    binary_targets = np.where(labels == label, 1, -1)
    x_train, x_test, y_train, y_test = train_test_split(
        data_set, binary_targets, test_size=0.2, random_state=15
    )
    return x_train, x_test, y_train, y_test


In [9]:
def get_labels_count(data_set):
    labels_count = len(np.unique(data_set))
    return labels_count


In [10]:
def train(train_data_set, train_labels, kernel_type='linear', C=10, threshold=1e-5):
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)

    P = train_labels * train_labels.transpose() * kernel
    q = -np.ones((objects_count, 1))
    G = np.concatenate((np.eye(objects_count), -np.eye(objects_count)))
    h = np.concatenate((C * np.ones((objects_count, 1)), np.zeros((objects_count, 1))))

    A = train_labels.reshape(1, objects_count)
    A = A.astype(float)
    b = 0.0

    sol = cvxopt.solvers.qp(cvxopt.matrix(P), cvxopt.matrix(q), cvxopt.matrix(G), cvxopt.matrix(h), cvxopt.matrix(A), cvxopt.matrix(b))

    lambdas = np.array(sol['x'])

    support_vectors_id = np.where(lambdas > threshold)[0]
    vector_number = len(support_vectors_id)
    support_vectors = train_data_set[support_vectors_id, :]

    lambdas = lambdas[support_vectors_id]
    targets = train_labels[support_vectors_id]

    b = np.sum(targets)
    for n in range(vector_number):
        b -= np.sum(lambdas * targets * np.reshape(kernel[support_vectors_id[n], support_vectors_id], (vector_number, 1)))
    b /= len(lambdas)

    return lambdas, support_vectors, support_vectors_id, b, targets, vector_number

def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    return kernel

def classify_rbf(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id):
    kernel = np.dot(test_data_set, support_vectors.T)
    sigma = 1.0
    #K = np.dot(test_data_set, support_vectors.T)
    #kernel = build_kernel(train_data_set, kernel_type='rbf')
    c = (1. / sigma * np.sum(test_data_set ** 2, axis=1) * np.ones((1, np.shape(test_data_set)[0]))).T
    c = np.dot(c, np.ones((1, np.shape(kernel)[1])))
    #aa = np.dot((np.diag(K)*np.ones((1,len(test_data_set)))).T[support_vectors_id], np.ones((1, np.shape(K)[0]))).T
    sv = (np.diag(np.dot(train_data_set, train_data_set.T))*np.ones((1,len(train_data_set)))).T[support_vectors_id]
    aa = np.dot(sv,np.ones((1,np.shape(kernel)[0]))).T
    kernel = kernel - 0.5 * c - 0.5 * aa
    kernel = np.exp(kernel / (2. * sigma ** 2))

    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
    return np.sign(y)

In [ ]:
# modify this part
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import cvxopt
import numpy as np

iris = load_iris()
data_set = iris.data
labels = iris.target

classes = np.unique(labels)
labels_count = get_labels_count(labels)

# one shared split for all one-vs-rest classifiers
x_train, x_test, y_train_multiclass, y_test_multiclass = train_test_split(
    data_set, labels, test_size=0.2, random_state=15, stratify=labels
)

models = []
for cls in classes:
    y_train_binary = np.where(y_train_multiclass == cls, 1, -1)
    objects_count = len(y_train_binary)
    lambdas, support_vectors, support_vectors_id, b, targets, vector_number = train(
        x_train, y_train_binary, kernel_type='rbf', C=0.1
    )
    models.append((cls, lambdas, targets, b, vector_number, support_vectors, support_vectors_id, x_train))

scores = np.zeros((len(x_test), labels_count))
for class_idx, model in enumerate(models):
    cls, lambdas, targets, b, vector_number, support_vectors, support_vectors_id, x_train_current = model

    kernel = np.dot(x_test, support_vectors.T)
    sigma = 1.0
    c = (1. / sigma * np.sum(x_test ** 2, axis=1) * np.ones((1, np.shape(x_test)[0]))).T
    c = np.dot(c, np.ones((1, np.shape(kernel)[1])))
    sv = (np.diag(np.dot(x_train_current, x_train_current.T)) * np.ones((1, len(x_train_current)))).T[support_vectors_id]
    aa = np.dot(sv, np.ones((1, np.shape(kernel)[0]))).T
    kernel = kernel - 0.5 * c - 0.5 * aa
    kernel = np.exp(kernel / (2. * sigma ** 2))

    decision = np.zeros((np.shape(x_test)[0], 1))
    for row_idx in range(np.shape(x_test)[0]):
        for sv_idx in range(vector_number):
            decision[row_idx] += lambdas[sv_idx] * targets[sv_idx] * kernel[row_idx, sv_idx]
        decision[row_idx] += b

    scores[:, class_idx] = decision.ravel()

predicted = classes[np.argmax(scores, axis=1)]
print(accuracy_score(predicted, y_test_multiclass))



     pcost       dcost       gap    pres   dres
 0: -1.6016e+00 -1.4697e+01  3e+02  2e+01  2e-16
 1: -1.3986e+00 -1.3213e+01  1e+01  4e-02  3e-16
 2: -1.4300e+00 -2.5122e+00  1e+00  4e-03  2e-16
 3: -1.5347e+00 -1.7989e+00  3e-01  4e-04  1e-16
 4: -1.5733e+00 -1.6928e+00  1e-01  1e-04  1e-16
 5: -1.5936e+00 -1.6308e+00  4e-02  3e-05  1e-16
 6: -1.6036e+00 -1.6111e+00  8e-03  1e-06  1e-16
 7: -1.6058e+00 -1.6076e+00  2e-03  2e-07  1e-16
 8: -1.6065e+00 -1.6066e+00  1e-04  1e-08  1e-16
 9: -1.6065e+00 -1.6065e+00  4e-06  3e-10  1e-16
10: -1.6065e+00 -1.6065e+00  5e-08  4e-12  2e-16
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0: -1.6937e+00 -1.4596e+01  3e+02  2e+01  2e-16
 1: -1.4357e+00 -1.3076e+01  2e+01  4e-01  3e-16
 2: -1.3158e+00 -2.9617e+00  2e+00  3e-16  3e-16
 3: -1.4499e+00 -1.9283e+00  5e-01  2e-16  2e-16
 4: -1.5137e+00 -1.7773e+00  3e-01  1e-16  2e-16
 5: -1.5617e+00 -1.6675e+00  1e-01  2e-16  2e-16
 6: -1.5879e+00 -1.6219e+00  3e-02  2e-16  2e-1